# Clinical Data and Trial Analytics Portfolio

This fully reproducible notebook demonstrates:

- clinical trial simulation and patient-level data generation;
- SDTM-style DM, EX, AE, LB, PC, and DS domain construction;
- ADaM-style ADSL, ADTTE, and ADRESP derivations;
- noncompartmental pharmacokinetic analysis;
- adverse-event and Grade 3 or higher safety reporting;
- Kaplan-Meier survival analysis and Cox proportional-hazards modelling;
- endpoint evaluation and biomarker integration;
- governed generative-AI support for programming, review, and interpretation.

All records are synthetic. The outputs are educational portfolio examples and are not submission-ready.

## 1. Environment and reproducibility

The workflow uses deterministic simulation, explicit derivations, validation checks, and separation of deterministic analyses from optional generative-AI support.

In [ ]:
# Uncomment in a fresh environment when required:
# %pip install pandas numpy scipy scikit-learn matplotlib lifelines

from __future__ import annotations

import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.model_selection import StratifiedKFold, cross_val_predict

warnings.filterwarnings("ignore")
SEED = 20260714
rng = np.random.default_rng(SEED)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

print("Environment ready.")

## 2. Simulate a randomised oncology trial

In [ ]:
N = 360
STUDY_ID = "NVS-AI-001"

subjects = pd.DataFrame({
    "STUDYID": STUDY_ID,
    "USUBJID": [f"{STUDY_ID}-{i:04d}" for i in range(1, N + 1)],
    "SUBJID": [f"{i:04d}" for i in range(1, N + 1)],
})

subjects["TRT01P"] = rng.choice(["CONTROL", "AI_GUIDED_TX"], N)
subjects["COUNTRY"] = rng.choice(["GBR", "USA", "CHE", "IRL"], N, p=[0.30, 0.35, 0.20, 0.15])
subjects["SITEID"] = [f"{c}-{rng.integers(1,5):02d}" for c in subjects["COUNTRY"]]
subjects["AGE"] = np.clip(rng.normal(61, 10, N), 28, 85).round().astype(int)
subjects["SEX"] = rng.choice(["F", "M"], N, p=[0.48, 0.52])
subjects["RACE"] = rng.choice(
    ["WHITE", "ASIAN", "BLACK OR AFRICAN AMERICAN", "OTHER"],
    N, p=[0.58, 0.23, 0.13, 0.06]
)
subjects["ECOG"] = rng.choice([0, 1, 2], N, p=[0.25, 0.57, 0.18])
subjects["STAGE"] = rng.choice(["III", "IV"], N, p=[0.26, 0.74])
subjects["BMI"] = np.clip(rng.normal(26.4, 4.8, N), 17, 43).round(1)
subjects["TUMOUR_BURDEN"] = np.exp(rng.normal(np.log(74), 0.40, N)).round(1)

biomarker_z = rng.normal(0, 1, N)
subjects["BIOMARKER"] = (50 + 17*biomarker_z + 4*(subjects["STAGE"] == "IV") + rng.normal(0, 4, N)).round(2)
subjects["BIOMARKER_POS"] = np.where(
    subjects["BIOMARKER"] >= subjects["BIOMARKER"].quantile(0.60), "Y", "N"
)

subjects["RANDDT"] = pd.Timestamp("2025-01-15") + pd.to_timedelta(rng.integers(0, 180, N), unit="D")
subjects.head()

In [ ]:
# Simulate overall survival and objective response.
trt = (subjects["TRT01P"] == "AI_GUIDED_TX").astype(int)
bmk = (subjects["BIOMARKER_POS"] == "Y").astype(int)

lp = (
    0.38*(subjects["ECOG"] == 2)
    + 0.45*(subjects["STAGE"] == "IV")
    + 0.007*(subjects["AGE"] - 60)
    + 0.0035*(subjects["TUMOUR_BURDEN"] - 70)
    - 0.22*bmk
    - 0.28*trt
    - 0.36*trt*bmk
)

hazard = (1/300) * np.exp(lp)
event_time = rng.exponential(1/hazard)
censor_time = rng.uniform(300, 540, N)

subjects["OS_DAYS"] = np.ceil(np.minimum(event_time, censor_time)).astype(int)
subjects["OS_EVENT"] = (event_time <= censor_time).astype(int)
subjects["OS_CNSR"] = 1 - subjects["OS_EVENT"]

response_logit = (
    -0.90 + 0.70*trt + 0.55*bmk + 0.75*trt*bmk
    - 0.45*(subjects["ECOG"] == 2)
    - 0.35*(subjects["STAGE"] == "IV")
)
subjects["ORR"] = rng.binomial(1, 1/(1 + np.exp(-response_logit)))
subjects["EOSDT"] = subjects["RANDDT"] + pd.to_timedelta(subjects["OS_DAYS"], unit="D")

subjects[["USUBJID", "TRT01P", "BIOMARKER_POS", "OS_DAYS", "OS_EVENT", "ORR"]].head()

In [ ]:
trial_checks = pd.Series({
    "USUBJID unique": subjects["USUBJID"].is_unique,
    "Treatment valid": set(subjects["TRT01P"]) <= {"CONTROL", "AI_GUIDED_TX"},
    "Follow-up non-negative": (subjects["OS_DAYS"] >= 0).all(),
    "Event binary": set(subjects["OS_EVENT"]) <= {0, 1},
    "Response binary": set(subjects["ORR"]) <= {0, 1},
    "Randomisation date complete": subjects["RANDDT"].notna().all(),
}, name="Passed")
trial_checks

## 3. Construct SDTM-style domains

In [ ]:
# DM, Demographics
dm = subjects[
    ["STUDYID", "USUBJID", "SUBJID", "SITEID", "COUNTRY", "AGE", "SEX", "RACE"]
].copy()
dm.insert(1, "DOMAIN", "DM")
dm["RFSTDTC"] = subjects["RANDDT"].dt.strftime("%Y-%m-%d")
dm["ARMCD"] = subjects["TRT01P"].map({"CONTROL": "CTRL", "AI_GUIDED_TX": "AITX"})
dm["ARM"] = subjects["TRT01P"]
dm["ACTARMCD"] = dm["ARMCD"]
dm["ACTARM"] = dm["ARM"]
dm["AGEU"] = "YEARS"
dm.head()

In [ ]:
# EX, Exposure
ex_rows = []
for _, r in subjects.iterrows():
    for cycle in range(1, 7):
        exstdt = r["RANDDT"] + pd.Timedelta(days=(cycle-1)*21)
        if exstdt > r["EOSDT"]:
            break
        base_dose = 100 if r["TRT01P"] == "CONTROL" else 120
        ex_rows.append({
            "STUDYID": r["STUDYID"], "DOMAIN": "EX", "USUBJID": r["USUBJID"],
            "EXSEQ": cycle, "EXTRT": r["TRT01P"],
            "EXDOSE": round(base_dose*rng.normal(1, 0.05), 1),
            "EXDOSU": "mg", "EXDOSFRM": "INFUSION",
            "EXSTDTC": exstdt.strftime("%Y-%m-%d"),
            "EXENDTC": (exstdt + pd.Timedelta(hours=1)).strftime("%Y-%m-%dT%H:%M")
        })
ex = pd.DataFrame(ex_rows)
ex.head()

In [ ]:
# AE, Adverse Events
ae_terms = [
    ("NAUSEA", "GASTROINTESTINAL DISORDERS"),
    ("FATIGUE", "GENERAL DISORDERS"),
    ("NEUTROPENIA", "BLOOD AND LYMPHATIC SYSTEM DISORDERS"),
    ("ANAEMIA", "BLOOD AND LYMPHATIC SYSTEM DISORDERS"),
    ("DIARRHOEA", "GASTROINTESTINAL DISORDERS"),
    ("RASH", "SKIN AND SUBCUTANEOUS TISSUE DISORDERS"),
    ("PYREXIA", "GENERAL DISORDERS"),
    ("THROMBOCYTOPENIA", "BLOOD AND LYMPHATIC SYSTEM DISORDERS"),
]

ae_rows = []
for _, r in subjects.iterrows():
    risk = (1.18 if r["TRT01P"] == "AI_GUIDED_TX" else 1.0)
    risk *= 1 + 0.22*(r["ECOG"] == 2) + 0.12*(r["AGE"] >= 70)
    for seq in range(1, rng.poisson(1.8*risk) + 1):
        term, soc = ae_terms[rng.integers(0, len(ae_terms))]
        max_day = max(2, min(int(r["OS_DAYS"]), 180))
        start_day = int(rng.integers(1, max_day))
        duration = int(rng.integers(1, 29))
        probs = np.array([0.35, 0.42, 0.18, 0.045, 0.005])
        if r["TRT01P"] == "AI_GUIDED_TX":
            probs += np.array([-0.02, -0.02, 0.025, 0.012, 0.003])
        probs = np.clip(probs, 0, None)
        probs /= probs.sum()
        grade = int(rng.choice([1,2,3,4,5], p=probs))
        ae_rows.append({
            "STUDYID": r["STUDYID"], "DOMAIN": "AE", "USUBJID": r["USUBJID"],
            "AESEQ": seq, "AETERM": term, "AEDECOD": term, "AEBODSYS": soc,
            "AESTDTC": (r["RANDDT"] + pd.Timedelta(days=start_day)).strftime("%Y-%m-%d"),
            "AEENDTC": (r["RANDDT"] + pd.Timedelta(days=start_day+duration)).strftime("%Y-%m-%d"),
            "AESEV": {1:"MILD",2:"MODERATE",3:"SEVERE",4:"LIFE THREATENING",5:"FATAL"}[grade],
            "AETOXGR": str(grade),
            "AESER": "Y" if grade >= 4 or rng.random() < 0.03 else "N",
            "AEREL": rng.choice(["NOT RELATED","POSSIBLY RELATED","RELATED"], p=[0.35,0.42,0.23]),
            "AETRTEM": "Y"
        })
ae = pd.DataFrame(ae_rows)
ae.head()

In [ ]:
# LB, Laboratory Test Results
tests = [
    ("HGB", "Haemoglobin", "g/dL", 13.4, 1.4),
    ("NEUT", "Neutrophils", "10^9/L", 4.2, 1.1),
    ("ALT", "Alanine aminotransferase", "U/L", 28, 11),
    ("CREAT", "Creatinine", "umol/L", 78, 16),
]
visits = [("BASELINE",0), ("CYCLE 2",21), ("CYCLE 4",63), ("END OF TREATMENT",126)]

lb_rows = []
for _, r in subjects.iterrows():
    for visit, day in visits:
        if day > r["OS_DAYS"]:
            continue
        for code_, name, unit, mean, sd in tests:
            shift = 0
            if visit != "BASELINE" and r["TRT01P"] == "AI_GUIDED_TX":
                shift = {"HGB":-0.5, "NEUT":-0.7, "ALT":7, "CREAT":3}[code_]
            value = max(0.1, rng.normal(mean+shift, sd))
            lb_rows.append({
                "STUDYID": r["STUDYID"], "DOMAIN": "LB", "USUBJID": r["USUBJID"],
                "LBTESTCD": code_, "LBTEST": name,
                "LBORRES": round(value,2), "LBORRESU": unit,
                "LBSTRESN": round(value,2), "LBSTRESU": unit,
                "VISIT": visit, "LBDY": day+1,
                "LBDTC": (r["RANDDT"] + pd.Timedelta(days=day)).strftime("%Y-%m-%d")
            })
lb = pd.DataFrame(lb_rows)
lb.head()

In [ ]:
# Compatible with pandas 2.x and 3.x.
# PC, Pharmacokinetic Concentrations
# Select an equal number of PK subjects per treatment while preserving
# the treatment column across pandas versions.
pk_subjects = pd.concat(
    [
        group.sample(
            n=min(36, len(group)),
            random_state=SEED + i
        )
        for i, (_, group) in enumerate(
            subjects.groupby("TRT01P", sort=True)
        )
    ],
    ignore_index=True
)

required_pk_columns = {"USUBJID", "TRT01P", "RANDDT"}
missing_pk_columns = required_pk_columns.difference(pk_subjects.columns)
if missing_pk_columns:
    raise KeyError(
        f"PK subject selection lost required columns: {sorted(missing_pk_columns)}"
    )

times = np.array([0,0.5,1,2,4,8,12,24], dtype=float)
pc_rows = []
for _, r in pk_subjects.iterrows():
    dose = 100 if r["TRT01P"] == "CONTROL" else 120
    ka = rng.lognormal(np.log(1.25), 0.22)
    ke = rng.lognormal(np.log(0.10), 0.20)
    vd = rng.lognormal(np.log(42), 0.18)
    conc = dose/vd * ka/(ka-ke) * (np.exp(-ke*times) - np.exp(-ka*times))
    conc[0] = 0
    conc *= rng.lognormal(0, 0.08, len(times))
    for seq, (t, c) in enumerate(zip(times, conc), 1):
        pc_rows.append({
            "STUDYID": r["STUDYID"], "DOMAIN": "PC", "USUBJID": r["USUBJID"],
            "PCSEQ": seq, "PCTESTCD": "DRUGX",
            "PCTEST": "Drug X plasma concentration",
            "PCORRES": round(float(c),4), "PCORRESU": "mg/L",
            "PCSTRESN": round(float(c),4), "PCSTRESU": "mg/L",
            "PCTPTNUM": t, "PCTPT": f"{t:g} HOURS POSTDOSE",
            "PCDTC": (r["RANDDT"] + pd.Timedelta(hours=float(t))).strftime("%Y-%m-%dT%H:%M")
        })
pc = pd.DataFrame(pc_rows)
pc.head()

In [ ]:
# DS, Disposition
ds_rows = []
for _, r in subjects.iterrows():
    ds_rows.extend([
        {
            "STUDYID": r["STUDYID"], "DOMAIN": "DS", "USUBJID": r["USUBJID"],
            "DSSEQ": 1, "DSTERM": "RANDOMIZED", "DSDECOD": "RANDOMIZED",
            "DSCAT": "PROTOCOL MILESTONE", "DSSTDTC": r["RANDDT"].strftime("%Y-%m-%d")
        },
        {
            "STUDYID": r["STUDYID"], "DOMAIN": "DS", "USUBJID": r["USUBJID"],
            "DSSEQ": 2,
            "DSTERM": "DEATH" if r["OS_EVENT"] else "STUDY COMPLETED/CENSORED",
            "DSDECOD": "DEATH" if r["OS_EVENT"] else "COMPLETED",
            "DSCAT": "DISPOSITION EVENT", "DSSTDTC": r["EOSDT"].strftime("%Y-%m-%d")
        }
    ])
ds = pd.DataFrame(ds_rows)

domain_inventory = pd.DataFrame({
    "Domain": ["DM","EX","AE","LB","PC","DS"],
    "Rows": [len(dm),len(ex),len(ae),len(lb),len(pc),len(ds)],
    "Subjects": [x["USUBJID"].nunique() for x in [dm,ex,ae,lb,pc,ds]]
})
domain_inventory

## 4. Derive ADaM-style analysis datasets

In [ ]:
first_ex = (
    ex.assign(EXSTDTC_DT=pd.to_datetime(ex["EXSTDTC"]))
      .groupby("USUBJID", as_index=False)["EXSTDTC_DT"].min()
      .rename(columns={"EXSTDTC_DT":"TRTSDT"})
)
last_ex = (
    ex.assign(EXENDTC_DT=pd.to_datetime(ex["EXENDTC"]))
      .groupby("USUBJID", as_index=False)["EXENDTC_DT"].max()
      .rename(columns={"EXENDTC_DT":"TRTEDT"})
)

adsl = subjects.merge(first_ex, on="USUBJID", how="left").merge(last_ex, on="USUBJID", how="left")
adsl["TRT01A"] = adsl["TRT01P"]
adsl["ITTFL"] = "Y"
adsl["SAFFL"] = np.where(adsl["USUBJID"].isin(ex["USUBJID"]), "Y", "N")
adsl["EFFFL"] = np.where(adsl["ORR"].notna(), "Y", "N")
grade3_subjects = set(ae.loc[pd.to_numeric(ae["AETOXGR"]) >= 3, "USUBJID"])
adsl["GR3AEFL"] = np.where(adsl["USUBJID"].isin(grade3_subjects), "Y", "N")
adsl["AGEGR1"] = pd.cut(adsl["AGE"], [0,64,74,200], labels=["<65","65-74",">=75"])
adsl["BMKGR1"] = np.where(adsl["BIOMARKER_POS"] == "Y", "POSITIVE", "NEGATIVE")
adsl["RANDDTC"] = adsl["RANDDT"].dt.strftime("%Y-%m-%d")
adsl["EOSDTC"] = adsl["EOSDT"].dt.strftime("%Y-%m-%d")

adsl = adsl[[
    "STUDYID","USUBJID","SUBJID","SITEID","COUNTRY",
    "TRT01P","TRT01A","TRTSDT","TRTEDT","ITTFL","SAFFL","EFFFL","GR3AEFL",
    "AGE","AGEGR1","SEX","RACE","ECOG","STAGE","BMI",
    "BIOMARKER","BIOMARKER_POS","BMKGR1","TUMOUR_BURDEN",
    "RANDDTC","EOSDTC","OS_DAYS","OS_EVENT","OS_CNSR","ORR"
]].sort_values("USUBJID").reset_index(drop=True)

adsl.head()

In [ ]:
# ADTTE, time-to-event analysis
adtte = adsl[[
    "STUDYID","USUBJID","TRT01P","TRT01A","ITTFL",
    "AGE","SEX","ECOG","STAGE","BIOMARKER","BIOMARKER_POS",
    "OS_DAYS","OS_CNSR"
]].copy()
adtte["PARAMCD"] = "OS"
adtte["PARAM"] = "Overall Survival"
adtte["AVAL"] = adtte["OS_DAYS"]
adtte["CNSR"] = adtte["OS_CNSR"]
adtte["EVNTDESC"] = np.where(adtte["CNSR"] == 0, "DEATH", "CENSORED")
adtte["STARTDT"] = pd.to_datetime(adsl["RANDDTC"])
adtte["ADT"] = adtte["STARTDT"] + pd.to_timedelta(adtte["AVAL"], unit="D")

# ADRESP, binary response analysis
adresp = adsl[[
    "STUDYID","USUBJID","TRT01P","TRT01A","ITTFL",
    "BIOMARKER","BIOMARKER_POS","ORR"
]].copy()
adresp["PARAMCD"] = "ORR"
adresp["PARAM"] = "Objective Response"
adresp["AVAL"] = adresp["ORR"]
adresp["AVALC"] = np.where(adresp["AVAL"] == 1, "RESPONDER", "NON-RESPONDER")

adtte.head(), adresp.head()

In [ ]:
validation = pd.DataFrame({
    "Check": [
        "ADSL has one record per subject",
        "ITT equals randomised population",
        "Safety subjects have exposure",
        "ADTTE AVAL is non-negative",
        "ADTTE CNSR is binary",
        "ADRESP AVAL is binary",
        "Planned and actual treatment are populated"
    ],
    "Passed": [
        adsl["USUBJID"].is_unique,
        adsl.query("ITTFL == 'Y'")["USUBJID"].nunique() == subjects["USUBJID"].nunique(),
        set(adsl.query("SAFFL == 'Y'")["USUBJID"]) <= set(ex["USUBJID"]),
        (adtte["AVAL"] >= 0).all(),
        set(adtte["CNSR"]) <= {0,1},
        set(adresp["AVAL"]) <= {0,1},
        adsl[["TRT01P","TRT01A"]].notna().all().all()
    ]
})
validation

## 5. Pharmacokinetic analysis

In [ ]:
def linear_up_log_down_auc(time, conc):
    auc = 0.0
    for i in range(1, len(time)):
        t1, t2 = time[i-1], time[i]
        c1, c2 = conc[i-1], conc[i]
        dt = t2 - t1
        if c2 >= c1 or c1 <= 0 or c2 <= 0:
            auc += 0.5*(c1+c2)*dt
        else:
            auc += (c1-c2)/np.log(c1/c2)*dt
    return float(auc)

pk_rows = []
for usubjid, g in pc.groupby("USUBJID"):
    g = g.sort_values("PCTPTNUM")
    t = g["PCTPTNUM"].to_numpy(float)
    c = g["PCSTRESN"].to_numpy(float)
    imax = int(np.argmax(c))
    terminal = g[g["PCSTRESN"] > 0].tail(4)
    slope, intercept, r, p, se = stats.linregress(
        terminal["PCTPTNUM"], np.log(terminal["PCSTRESN"])
    )
    lambda_z = max(1e-9, -slope)
    auc024 = linear_up_log_down_auc(t, c)
    pk_rows.append({
        "USUBJID": usubjid,
        "CMAX": c[imax],
        "TMAX": t[imax],
        "AUC0_24": auc024,
        "LAMBDA_Z": lambda_z,
        "HALF_LIFE": np.log(2)/lambda_z,
        "AUCINF": auc024 + c[-1]/lambda_z,
        "R2_TERMINAL": r**2
    })

pk = pd.DataFrame(pk_rows).merge(
    adsl[["USUBJID","TRT01A","AGE","SEX","BMI","BIOMARKER_POS"]],
    on="USUBJID", how="left"
)
pk.head()

In [ ]:
pk_summary = (
    pk.groupby("TRT01A")
      .agg(
          N=("USUBJID","nunique"),
          CMAX_GMEAN=("CMAX", lambda x: np.exp(np.mean(np.log(x)))),
          CMAX_CV_PCT=("CMAX", lambda x: 100*np.std(x,ddof=1)/np.mean(x)),
          TMAX_MEDIAN=("TMAX","median"),
          AUC0_24_GMEAN=("AUC0_24", lambda x: np.exp(np.mean(np.log(x)))),
          HALF_LIFE_MEAN=("HALF_LIFE","mean")
      ).round(3)
)
pk_summary

In [ ]:
profile = (
    pc.merge(adsl[["USUBJID","TRT01A"]], on="USUBJID")
      .groupby(["TRT01A","PCTPTNUM"])["PCSTRESN"]
      .agg(["mean","std","count"]).reset_index()
)
profile["se"] = profile["std"]/np.sqrt(profile["count"])

plt.figure(figsize=(8,5))
for trt_name, g in profile.groupby("TRT01A"):
    plt.errorbar(g["PCTPTNUM"], g["mean"], yerr=g["se"], marker="o", capsize=3, label=trt_name)
plt.xlabel("Time after dose, hours")
plt.ylabel("Mean plasma concentration, mg/L")
plt.title("Mean concentration-time profiles")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Adverse-event summaries and Grade 3 or higher reporting

In [ ]:
safety_denoms = adsl.query("SAFFL == 'Y'").groupby("TRT01A")["USUBJID"].nunique()
ae_te = ae.merge(adsl[["USUBJID","TRT01A","SAFFL"]], on="USUBJID").query(
    "AETRTEM == 'Y' and SAFFL == 'Y'"
)

ae_summary = (
    ae_te.groupby(["TRT01A","AEDECOD"])["USUBJID"]
         .nunique().rename("n").reset_index()
)
ae_summary["N"] = ae_summary["TRT01A"].map(safety_denoms)
ae_summary["Percent"] = 100*ae_summary["n"]/ae_summary["N"]
ae_summary.sort_values(["TRT01A","Percent"], ascending=[True,False]).head(16)

In [ ]:
grade3 = ae_te[pd.to_numeric(ae_te["AETOXGR"]) >= 3].copy()

grade3_summary = (
    grade3.groupby(["TRT01A","AEDECOD"])["USUBJID"]
          .nunique().rename("n").reset_index()
)
grade3_summary["N"] = grade3_summary["TRT01A"].map(safety_denoms)
grade3_summary["Percent"] = 100*grade3_summary["n"]/grade3_summary["N"]

grade3_overall = (
    grade3.groupby("TRT01A")["USUBJID"].nunique()
          .rename("Subjects_with_Grade3plus").to_frame()
          .join(safety_denoms.rename("N"))
)
grade3_overall["Percent"] = 100*grade3_overall["Subjects_with_Grade3plus"]/grade3_overall["N"]

display(grade3_overall.round(2))
display(grade3_summary.sort_values(["TRT01A","Percent"], ascending=[True,False]).head(16))

In [ ]:
# Relative risk of any Grade 3+ TEAE
subject_grade3 = (
    adsl.query("SAFFL == 'Y'")[["USUBJID","TRT01A"]]
        .assign(GRADE3PLUS=lambda d: d["USUBJID"].isin(set(grade3["USUBJID"])).astype(int))
)
risk = subject_grade3.groupby("TRT01A")["GRADE3PLUS"].agg(["sum","count","mean"])
ctrl, tx = risk.loc["CONTROL"], risk.loc["AI_GUIDED_TX"]
rr = tx["mean"]/ctrl["mean"]
se_log_rr = math.sqrt((1/tx["sum"] - 1/tx["count"]) + (1/ctrl["sum"] - 1/ctrl["count"]))
ci = np.exp(np.log(rr) + np.array([-1,1])*1.96*se_log_rr)

pd.Series({
    "Risk_Control": ctrl["mean"],
    "Risk_AI_Guided": tx["mean"],
    "Relative_Risk": rr,
    "RR_95CI_Lower": ci[0],
    "RR_95CI_Upper": ci[1]
}).round(3)

## 7. Kaplan-Meier survival analysis

In [ ]:
def kaplan_meier(time, event):
    time = np.asarray(time)
    event = np.asarray(event)
    event_times = np.sort(np.unique(time[event == 1]))
    survival = 1.0
    rows = [{"time":0.0, "survival":1.0, "n_risk":len(time), "n_event":0}]
    for t in event_times:
        n_risk = np.sum(time >= t)
        n_event = np.sum((time == t) & (event == 1))
        survival *= 1 - n_event/n_risk
        rows.append({"time":float(t), "survival":survival, "n_risk":int(n_risk), "n_event":int(n_event)})
    return pd.DataFrame(rows)

km_curves = {}
plt.figure(figsize=(8,5))
for trt_name, g in adtte.groupby("TRT01A"):
    curve = kaplan_meier(g["AVAL"], 1-g["CNSR"])
    km_curves[trt_name] = curve
    plt.step(curve["time"], curve["survival"], where="post", label=trt_name)

plt.xlabel("Time from randomisation, days")
plt.ylabel("Overall survival probability")
plt.title("Kaplan-Meier overall survival")
plt.ylim(0,1.02)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def median_survival(curve):
    reached = curve.loc[curve["survival"] <= 0.5, "time"]
    return float(reached.iloc[0]) if len(reached) else np.nan

pd.Series({name: median_survival(curve) for name, curve in km_curves.items()},
          name="Median_OS_days")

## 8. Cox proportional-hazards modelling

In [ ]:
cox_data = adsl[[
    "OS_DAYS","OS_EVENT","TRT01A","AGE","SEX","ECOG",
    "STAGE","BIOMARKER_POS","TUMOUR_BURDEN"
]].copy()
cox_data["TRT_AI"] = (cox_data["TRT01A"] == "AI_GUIDED_TX").astype(int)
cox_data["SEX_M"] = (cox_data["SEX"] == "M").astype(int)
cox_data["STAGE_IV"] = (cox_data["STAGE"] == "IV").astype(int)
cox_data["BIOMARKER_POS_I"] = (cox_data["BIOMARKER_POS"] == "Y").astype(int)
cox_data["TRT_BIOMARKER_INT"] = cox_data["TRT_AI"]*cox_data["BIOMARKER_POS_I"]

model_cols = [
    "OS_DAYS","OS_EVENT","TRT_AI","AGE","SEX_M","ECOG",
    "STAGE_IV","BIOMARKER_POS_I","TUMOUR_BURDEN","TRT_BIOMARKER_INT"
]

try:
    from lifelines import CoxPHFitter
    cph = CoxPHFitter()
    cph.fit(cox_data[model_cols], duration_col="OS_DAYS", event_col="OS_EVENT")
    cox_results = cph.summary[
        ["coef","exp(coef)","se(coef)","p","exp(coef) lower 95%","exp(coef) upper 95%"]
    ]
    display(cox_results.round(4))
except ImportError:
    print("Install lifelines to run this cell: %pip install lifelines")

In [ ]:
try:
    cph.check_assumptions(cox_data[model_cols], p_value_threshold=0.05, show_plots=False)
except Exception as exc:
    print("Proportional-hazards diagnostic skipped or reported:", exc)

## 9. Endpoint evaluation and biomarker integration

In [ ]:
orr = (
    adsl.groupby(["TRT01A","BIOMARKER_POS"])["ORR"]
        .agg(["sum","count","mean"]).reset_index()
)
orr["Percent"] = 100*orr["mean"]
orr

In [ ]:
def risk_difference_ci(x1, n1, x0, n0):
    p1, p0 = x1/n1, x0/n0
    rd = p1-p0
    se = math.sqrt(p1*(1-p1)/n1 + p0*(1-p0)/n0)
    return rd, rd-1.96*se, rd+1.96*se

effects = []
for status, g in adsl.groupby("BIOMARKER_POS"):
    tab = g.groupby("TRT01A")["ORR"].agg(["sum","count"])
    rd, lo, hi = risk_difference_ci(
        tab.loc["AI_GUIDED_TX","sum"], tab.loc["AI_GUIDED_TX","count"],
        tab.loc["CONTROL","sum"], tab.loc["CONTROL","count"]
    )
    effects.append({
        "BIOMARKER_POS": status,
        "Risk_difference": rd,
        "Lower_95": lo,
        "Upper_95": hi
    })

pd.DataFrame(effects).set_index("BIOMARKER_POS").round(3)

In [ ]:
# Cross-validated response model
X = pd.DataFrame({
    "TRT_AI": (adsl["TRT01A"] == "AI_GUIDED_TX").astype(int),
    "AGE": adsl["AGE"],
    "ECOG": adsl["ECOG"],
    "STAGE_IV": (adsl["STAGE"] == "IV").astype(int),
    "BIOMARKER": adsl["BIOMARKER"],
    "TUMOUR_BURDEN": adsl["TUMOUR_BURDEN"],
})
X["TRT_BIOMARKER"] = X["TRT_AI"]*X["BIOMARKER"]
y = adsl["ORR"]

cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
pred = cross_val_predict(
    LogisticRegression(max_iter=2000), X, y, cv=cv, method="predict_proba"
)[:,1]

pd.Series({
    "ROC_AUC": roc_auc_score(y, pred),
    "Brier_score": brier_score_loss(y, pred),
    "Observed_response_rate": y.mean(),
    "Mean_predicted_probability": pred.mean()
}).round(4)

In [ ]:
calibration = pd.DataFrame({"observed":y, "predicted":pred})
calibration["decile"] = pd.qcut(calibration["predicted"], 10, duplicates="drop")
calibration_table = calibration.groupby("decile", observed=False).agg(
    mean_predicted=("predicted","mean"),
    observed_rate=("observed","mean"),
    n=("observed","size")
).reset_index(drop=True)

plt.figure(figsize=(6,5))
plt.plot([0,1],[0,1], linestyle="--")
plt.plot(calibration_table["mean_predicted"], calibration_table["observed_rate"], marker="o")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed response rate")
plt.title("Response-model calibration")
plt.tight_layout()
plt.show()

calibration_table.round(3)

## 10. Governed generative-AI support

In [ ]:
def aggregate_review_summary():
    return {
        "study_id": STUDY_ID,
        "subject_count": int(dm["USUBJID"].nunique()),
        "domain_rows": {
            "DM": len(dm), "EX": len(ex), "AE": len(ae),
            "LB": len(lb), "PC": len(pc), "DS": len(ds)
        },
        "analysis_rows": {
            "ADSL": len(adsl), "ADTTE": len(adtte), "ADRESP": len(adresp)
        },
        "validation_failures": validation.loc[~validation["Passed"], "Check"].tolist(),
        "missing_values": {
            "DM": int(dm.isna().sum().sum()),
            "AE": int(ae.isna().sum().sum()),
            "ADSL": int(adsl.isna().sum().sum())
        },
        "safety": {
            "subjects_with_grade3plus_teae": int(grade3["USUBJID"].nunique()),
            "subjects_with_serious_ae": int(ae_te.loc[ae_te["AESER"] == "Y","USUBJID"].nunique())
        },
        "efficacy": {
            "overall_response_rate": float(adsl["ORR"].mean()),
            "deaths": int(adsl["OS_EVENT"].sum()),
            "censored": int(adsl["OS_CNSR"].sum())
        }
    }

review_summary = aggregate_review_summary()
print(json.dumps(review_summary, indent=2))

In [ ]:
SYSTEM_GUARDRAILS = (
    "You are a clinical-programming review assistant in a regulated environment. "
    "Use only supplied aggregate, de-identified data. Do not infer patient-level facts. "
    "Separate observations, possible explanations, and recommended deterministic checks. "
    "State uncertainty and require human programmer and statistician review."
)

def build_review_prompt(summary):
    return (
        SYSTEM_GUARDRAILS
        + "\n\nTASK:\n"
        + "1. Summarise data quality.\n"
        + "2. Identify safety or efficacy findings requiring verification.\n"
        + "3. Propose traceable programming checks.\n"
        + "4. Provide a plain-language scientific interpretation.\n"
        + "5. State limitations.\n\n"
        + "AGGREGATE SUMMARY:\n"
        + json.dumps(summary, indent=2)
    )

review_prompt = build_review_prompt(review_summary)
print(review_prompt[:2500])

In [ ]:
def deterministic_review(summary):
    lines = []
    if summary["validation_failures"]:
        lines.append("Validation failures: " + ", ".join(summary["validation_failures"]))
    else:
        lines.append("All programmed structural validation checks passed.")
    lines.append(
        f'{summary["safety"]["subjects_with_grade3plus_teae"]} subjects had at least one '
        "Grade 3 or higher treatment-emergent adverse event."
    )
    lines.append(
        f'The simulated overall response rate was '
        f'{100*summary["efficacy"]["overall_response_rate"]:.1f}%.'
    )
    lines.append(
        "All findings require confirmation against source domains, derivation specifications, "
        "analysis flags, denominators, censoring rules, and the statistical analysis plan."
    )
    return "\n".join("- " + x for x in lines)

print(deterministic_review(review_summary))

### Production controls for generative AI

A governed enterprise implementation should include role-based access, protected-data filtering, model-version control, prompt and response logging, approved metadata grounding, deterministic verification, human sign-off, audit trails, prompt-injection testing, and monitoring for drift and hallucination.

## 11. Study scorecard and exports

In [ ]:
scorecard = pd.DataFrame({
    "Metric": [
        "Randomised subjects",
        "Safety population",
        "Overall response rate, %",
        "Deaths",
        "Subjects with Grade 3+ TEAE",
        "PK-evaluable subjects",
        "SDTM-style domains",
        "ADaM-style datasets"
    ],
    "Value": [
        adsl["USUBJID"].nunique(),
        adsl.query("SAFFL == 'Y'")["USUBJID"].nunique(),
        round(100*adsl["ORR"].mean(),1),
        int(adsl["OS_EVENT"].sum()),
        int(grade3["USUBJID"].nunique()),
        int(pk["USUBJID"].nunique()),
        6,
        3
    ]
})
scorecard

In [ ]:
output_dir = Path("clinical_trial_analytics_outputs")
output_dir.mkdir(exist_ok=True)

exports = {
    "dm.csv": dm, "ex.csv": ex, "ae.csv": ae, "lb.csv": lb,
    "pc.csv": pc, "ds.csv": ds, "adsl.csv": adsl,
    "adtte.csv": adtte, "adresp.csv": adresp,
    "pk_parameters.csv": pk, "ae_summary.csv": ae_summary,
    "grade3plus_ae_summary.csv": grade3_summary,
    "scorecard.csv": scorecard
}

for filename, df in exports.items():
    df.to_csv(output_dir/filename, index=False)

with open(output_dir/"aggregate_ai_review_summary.json", "w", encoding="utf-8") as f:
    json.dump(review_summary, f, indent=2)

print(f"Exported {len(exports)+1} files to {output_dir.resolve()}")

## Conclusion

The notebook connects clinical data standards, analysis-ready derivations, pharmacokinetics, safety, survival, efficacy, biomarkers, predictive modelling, validation, and governed generative-AI assistance in one traceable workflow. In production, every derivation and analysis would be aligned with approved protocol, data-management plan, metadata standards, and statistical analysis plan, with independent quality control.